In [1]:
import math
from dataclasses import dataclass
from datetime import datetime
from typing import Literal, Optional

import yfinance as yf
from scipy.stats import norm
from scipy.optimize import brentq


# ========= 1. 数据结构 =========


@dataclass
class FuturesInput:
    S0: float
    r: float
    q: float
    T: float


@dataclass
class FuturesOptionInput:
    F0: float
    K: float
    r: float
    T: float
    sigma: float
    option_type: Literal["call", "put"] = "call"


# ========= 2. A50 期货理论价 =========


def theoretical_futures_price(inp: FuturesInput) -> float:
    return inp.S0 * math.exp((inp.r - inp.q) * inp.T)


# ========= 3. Black-76 期货期权定价 =========


def black76_price(inp: FuturesOptionInput) -> float:
    F0, K, r, T, sigma = inp.F0, inp.K, inp.r, inp.T, inp.sigma

    if T <= 0 or sigma <= 0:
        intrinsic = max(0.0, (F0 - K) if inp.option_type == "call" else (K - F0))
        return math.exp(-r * T) * intrinsic

    d1 = (math.log(F0 / K) + 0.5 * sigma * sigma * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)

    if inp.option_type == "call":
        price = math.exp(-r * T) * (F0 * norm.cdf(d1) - K * norm.cdf(d2))
    else:
        price = math.exp(-r * T) * (K * norm.cdf(-d2) - F0 * norm.cdf(-d1))

    return price


def black76_implied_vol(
    market_price: float,
    F0: float,
    K: float,
    r: float,
    T: float,
    option_type: Literal["call", "put"] = "call",
    sigma_low: float = 1e-4,
    sigma_high: float = 5.0,
    tol: float = 1e-6,
    max_iter: int = 100,
) -> Optional[float]:
    def f(sigma: float) -> float:
        inp = FuturesOptionInput(
            F0=F0, K=K, r=r, T=T, sigma=sigma, option_type=option_type
        )
        return black76_price(inp) - market_price

    try:
        return brentq(f, sigma_low, sigma_high, xtol=tol, maxiter=max_iter)
    except ValueError:
        return None


# ========= 4. 用 yfinance 获取 A50 指数现价 =========
# Yahoo Finance 上 FTSE China A50 Index 的代码示例：XIN9.FGI。[web:48][web:59][web:62][web:67][web:65]


def get_a50_spot_from_yahoo(ticker: str = "XIN9.FGI") -> float:
    """
    用 yfinance 获取 A50 指数最新收盘价/现价。
    """
    data = yf.Ticker(ticker)
    hist = data.history(period="1d")
    if hist.empty:
        raise ValueError(f"No data for ticker {ticker}")
    # 你可以用 'Close' 或 'Adj Close'
    return float(hist["Close"].iloc[-1])


# ========= 5. 示例主程序 =========

if __name__ == "__main__":
    # 1) 获取 A50 指数现货 S0
    S0 = get_a50_spot_from_yahoo("XIN9.FGI")
    print(f"当前 A50 指数现价(来自 Yahoo): {S0:.2f}")

    # 2) 设置期货/期权参数
    # 这里手动填：到期日、无风险利率 r、股息率 q
    # 假设到期日 2026-03-31
    today = datetime.today()
    expiry = datetime(2025, 12, 30)
    T = (expiry - today).days / 365.0

    r = 0.016  # 你可以换成 3M Shibor 或国债收益率
    q = 0.025  # 指数预估股息率

    # 3) 计算 A50 期货理论价格
    fut_inp = FuturesInput(S0=S0, r=r, q=q, T=T)
    F0_theoretical = theoretical_futures_price(fut_inp)
    print(f"A50 期货理论价格 (持有成本模型): {F0_theoretical:.2f}")

    # 4) 若你有实际期货价格，可以手动填入，或者以后接交易所 / broker API
    F0_market = 15200  # 这里先假设等于理论价

    # 5) Black-76 定价：假设一个期权
    K = F0_market  # ATM 执行价
    sigma = 0.25  # 年化波动率假设
    call_input = FuturesOptionInput(
        F0=F0_market, K=K, r=r, T=T, sigma=sigma, option_type="call"
    )
    call_price = black76_price(call_input)
    print(f"A50 期货 ATM 看涨期权理论价格: {call_price:.2f}")

    # 6) 如果你有市场期权价格，可以反解隐含波动率
    market_call_price = call_price  # 示例中先用相同值
    iv = black76_implied_vol(
        market_price=market_call_price, F0=F0_market, K=K, r=r, T=T, option_type="call"
    )
    print(f"隐含波动率: {iv:.4%}" if iv is not None else "隐含波动率求解失败")


当前 A50 指数现价(来自 Yahoo): 15165.96
A50 期货理论价格 (持有成本模型): 15161.85
A50 期货 ATM 看涨期权理论价格: 263.03
隐含波动率: 25.0000%
